# 🗃️ Manejo de datos estructurados
### Tipos, operaciones y práctica 

**Objetivos**
1. Describir qué es un dato estructurado y cuáles son sus **tipos** en una tabla.
2. Conocer las **operaciones** que se pueden realizar sobre ellos (inspeccionar, seleccionar, limpiar, transformar, combinar, agrupar, reestructurar, exportar).
3. Aplicar lo aprendido con **7 ejercicios** con autocorrección.

**Fuente de datos:** un archivo Excel (`datos_estructurados.xlsx`) con tres tablas relacionadas, que el notebook genera automáticamente.

**Cómo usarlo:** ejecuta las celdas en orden (`Run ▸ Run All Cells`). Los ejercicios están al final.

## 🗺️ Agenda

| # | Sección |
|---|---|
| 1 | ¿Qué es un dato estructurado? |
| 2 | Tipos de datos estructurados |
| 3 | Los datos del notebook (modelo relacional) |
| 4 | Operaciones: inspección · selección · orden · transformación · limpieza · combinación · agregación · reestructuración · series · exportación |
| 5 | Equivalencias pandas ↔ SQL ↔ Excel |
| 6 | Ejercicios (7) |

# 1 · ¿Qué es un dato estructurado?

Un dato es **estructurado** cuando se organiza en un **esquema fijo y conocido de antemano**:

| Elemento | Significado | Ejemplo |
|---|---|---|
| **Tabla** | Conjunto de registros del mismo tipo | `clientes` |
| **Fila (registro)** | Una entidad u observación | un cliente |
| **Columna (atributo)** | Una característica, con **un único tipo** | `edad` |
| **Clave primaria (PK)** | Columna que identifica de forma única cada fila | `id_cliente` |
| **Clave foránea (FK)** | Columna que referencia la PK de otra tabla | `pedidos.id_cliente` |

**Dónde viven:** hojas de Excel, archivos CSV, bases de datos relacionales (MySQL, PostgreSQL), DataFrames.

**Ventajas:** fáciles de consultar, validar y agregar. **Limitación:** poco flexibles si los datos cambian de forma.

# 2 · Tipos de datos estructurados

Cada columna tiene **un tipo**. En pandas (`dtype`) los principales son:

| Familia | dtype en pandas | Ejemplo | Operaciones típicas |
|---|---|---|---|
| Entero | `int64`, `Int64` (admite nulos) | `cantidad` | suma, media, conteo |
| Decimal | `float64` | `precio_unitario` | estadísticos, redondeo |
| Texto | `object` / `string` / `str` | `nombre` | `.str.strip()`, `.str.contains()` |
| Categórico | `category` | `segmento` | agrupar, ordenar por categoría |
| Booleano | `bool` | `activo` | filtros, conteos |
| Fecha y hora | `datetime64` | `fecha_pedido` | `.dt.month`, rangos, diferencias |
| Duración | `timedelta64` | 3 días | sumar a fechas, promediar |
| Nulo | `NaN`, `NaT`, `<NA>` | edad faltante | `isna()`, `fillna()` |

> Además del tipo **técnico** (dtype), está el tipo **estadístico**: nominal, ordinal, discreto, continuo… (ver notebook anterior).

In [ ]:
# %pip install -q pandas numpy matplotlib openpyxl

import os, json, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 200)
plt.rcParams.update({"figure.figsize": (9, 4), "axes.grid": True, "grid.alpha": .3,
                     "axes.spines.top": False, "axes.spines.right": False, "axes.titleweight": "bold"})
print("pandas", pd.__version__)

In [ ]:
# Un ejemplo mínimo de cada tipo, con su dtype explícito
tipos_demo = pd.DataFrame({
    "entero":      pd.Series([1, 2, 3], dtype="int64"),
    "decimal":     [1.5, 2.5, 3.5],
    "texto":       pd.Series(["a", "b", "c"], dtype="string"),
    "categoria":   pd.Categorical(["bajo", "alto", "bajo"], categories=["bajo", "medio", "alto"], ordered=True),
    "booleano":    [True, False, True],
    "fecha":       pd.to_datetime(["2025-01-01", "2025-02-01", "2025-03-01"]),
    "duracion":    pd.to_timedelta([1, 2, 3], unit="D"),
    "entero_nulo": pd.array([1, None, 3], dtype="Int64"),
})
display(tipos_demo)
display(tipos_demo.dtypes.to_frame("dtype"))

# 3 · Los datos del notebook

Simulamos el negocio de una cafetería con **3 tablas relacionadas**:

```
clientes (PK id_cliente) ──┐
                           ├──< pedidos (FK id_cliente, FK id_producto)
productos (PK id_producto) ─┘
```

El Excel contiene **imperfecciones a propósito** (nulos, duplicados, texto sucio, fechas como texto) para practicar limpieza.

In [ ]:
ARCHIVO_EXCEL = "datos_estructurados.xlsx"


def generar_excel(ruta):
    rng = np.random.default_rng(7)
    ciudades = ["Quito", "Guayaquil", "Ambato", "Cuenca", "Loja", "Riobamba", "Latacunga", "Píllaro"]

    # ---- clientes (con suciedad)
    n_c = 500
    c = pd.DataFrame({
        "id_cliente": np.arange(1, n_c + 1),
        "nombre": [f"Cliente {i:03d}" for i in range(1, n_c + 1)],
        "ciudad": rng.choice(ciudades, n_c),
        "edad": rng.integers(18, 70, n_c).astype(float),
        "email": [f"cliente{i}@correo.ec" for i in range(1, n_c + 1)],
        "segmento": rng.choice(["Básico", "Plus", "Premium"], n_c, p=[.55, .30, .15]),
        "fecha_registro": (pd.to_datetime("2024-01-01") + pd.to_timedelta(rng.integers(0, 600, n_c), unit="D")).strftime("%d/%m/%Y"),
    })
    ruido = rng.random(n_c)
    m1, m2 = ruido < .10, (ruido >= .10) & (ruido < .18)
    c.loc[m1, "ciudad"] = c.loc[m1, "ciudad"].str.lower() + " "
    c.loc[m2, "ciudad"] = c.loc[m2, "ciudad"].str.upper()
    c.loc[rng.random(n_c) < .07, "edad"] = np.nan
    c.loc[rng.random(n_c) < .10, "email"] = np.nan
    c = pd.concat([c, c.sample(8, random_state=1)], ignore_index=True)      # 8 duplicados exactos

    # ---- productos
    nombres = {
        "Bebidas": ["Café americano", "Café latte", "Té verde", "Té chai", "Jugo de naranja", "Jugo de mora",
                    "Batido de fresa", "Limonada", "Agua mineral", "Chocolate caliente"],
        "Postres": ["Brownie", "Cheesecake", "Helado artesanal", "Tres leches", "Flan", "Torta de zanahoria",
                    "Mousse de chocolate", "Churros", "Galleta gigante", "Pie de limón"],
        "Platos fuertes": ["Hamburguesa clásica", "Pizza personal", "Lasaña", "Pollo a la plancha", "Ceviche",
                           "Seco de pollo", "Tacos mixtos", "Ensalada César", "Sándwich club", "Pasta alfredo"],
    }
    rango = {"Bebidas": (1.5, 4.5), "Postres": (2.5, 6.0), "Platos fuertes": (6.0, 15.0)}
    filas, pid = [], 1
    for cat, lista in nombres.items():
        for nom in lista:
            precio = round(float(rng.uniform(*rango[cat])), 2)
            filas.append({"id_producto": pid, "producto": nom, "categoria": cat, "precio_unitario": precio,
                          "costo_unitario": round(precio * float(rng.uniform(.35, .65)), 2),
                          "activo": bool(rng.random() > .10)})
            pid += 1
    productos = pd.DataFrame(filas)

    # ---- pedidos
    n_p = 3000
    p = pd.DataFrame({
        "id_pedido": np.arange(10001, 10001 + n_p),
        "id_cliente": rng.integers(1, n_c + 1, n_p),
        "id_producto": rng.integers(1, 31, n_p),
        "fecha_pedido": pd.to_datetime("2025-01-01") + pd.to_timedelta(rng.integers(0, 365, n_p), unit="D"),
        "cantidad": rng.integers(1, 7, n_p),
        "descuento": rng.choice([0.0, 0.05, 0.10, 0.15, 0.20, 0.30], n_p, p=[.40, .15, .15, .10, .10, .10]),
        "metodo_pago": rng.choice(["Tarjeta", "Efectivo", "Transferencia"], n_p, p=[.5, .3, .2]),
        "estado": rng.choice(["Entregado", "Cancelado", "Pendiente"], n_p, p=[.8, .1, .1]),
    })
    p.loc[rng.random(n_p) < .15, "descuento"] = np.nan                       # descuento faltante
    p.loc[[10, 500, 1500], "cantidad"] = [45, 60, 52]                         # valores atípicos

    with pd.ExcelWriter(ruta, engine="openpyxl") as w:
        c.to_excel(w, sheet_name="clientes", index=False)
        productos.to_excel(w, sheet_name="productos", index=False)
        p.to_excel(w, sheet_name="pedidos", index=False)


if not os.path.exists(ARCHIVO_EXCEL):
    generar_excel(ARCHIVO_EXCEL)
    print("✅ Excel generado:", ARCHIVO_EXCEL)
else:
    print("📗 Usando Excel existente:", ARCHIVO_EXCEL)

hojas = pd.read_excel(ARCHIVO_EXCEL, sheet_name=None)
clientes, productos, pedidos = hojas["clientes"], hojas["productos"], hojas["pedidos"]
print({k: v.shape for k, v in hojas.items()})

In [ ]:
for nombre, df in [("clientes", clientes), ("productos", productos), ("pedidos", pedidos)]:
    print(f"\n=== {nombre} ===")
    display(df.head(4))

# 4 · Operaciones sobre datos estructurados

| Familia | Pregunta que responde | Herramientas en pandas |
|---|---|---|
| 4.1 Inspección | ¿Qué tengo? | `head`, `shape`, `info`, `describe`, `isna` |
| 4.2 Selección y filtrado | ¿Qué filas/columnas necesito? | `[]`, `loc`, `iloc`, máscaras, `query`, `isin` |
| 4.3 Ordenamiento | ¿Cuáles son los mayores/menores? | `sort_values`, `nlargest`, `rank` |
| 4.4 Transformación | ¿Cómo creo o convierto columnas? | `astype`, `map`, `apply`, `np.where`, `pd.cut`, `.str`, `.dt` |
| 4.5 Limpieza | ¿Cómo arreglo errores? | `fillna`, `dropna`, `drop_duplicates`, IQR |
| 4.6 Combinación | ¿Cómo uno tablas? | `merge`, `concat` |
| 4.7 Agregación | ¿Cuánto por grupo? | `groupby`, `agg`, `value_counts` |
| 4.8 Reestructuración | ¿Cómo cambio la forma? | `pivot_table`, `crosstab`, `melt` |
| 4.9 Series y ventanas | ¿Cómo evoluciona? | `resample`, `rolling`, `cumsum`, `pct_change` |
| 4.10 Exportación | ¿Cómo guardo el resultado? | `to_excel`, `to_csv` |

## 4.1 Inspección
Lo primero siempre: **conocer la forma, los tipos y los vacíos** de la tabla.

In [ ]:
print("Forma (filas, columnas):", pedidos.shape)
pedidos.info()
display(pedidos.describe().round(2))
display(clientes.isna().sum().to_frame("nulos").assign(**{"%": lambda d: (d["nulos"] / len(clientes) * 100).round(1)}))

## 4.2 Selección y filtrado
Elegir **columnas** (qué atributos) y **filas** (qué registros). Las condiciones se combinan con `&` (y), `|` (o), `~` (no).

In [ ]:
display(pedidos[["id_pedido", "cantidad"]].head(3))                                # columnas
display(pedidos.loc[pedidos["estado"] == "Entregado", ["id_pedido", "metodo_pago"]].head(3))   # filas por condición
display(pedidos.iloc[0:3, 0:3])                                                    # por posición

filtro = (pedidos["cantidad"] >= 5) & (pedidos["metodo_pago"].isin(["Tarjeta", "Transferencia"]))
print("Pedidos con cantidad >= 5 pagados con tarjeta o transferencia:", int(filtro.sum()))
print("Mismo resultado con query:",
      len(pedidos.query("cantidad >= 5 and metodo_pago in ['Tarjeta', 'Transferencia']")))

## 4.3 Ordenamiento y ranking

In [ ]:
display(pedidos.sort_values(["cantidad", "fecha_pedido"], ascending=[False, True]).head(5))
display(pedidos.nlargest(3, "cantidad")[["id_pedido", "cantidad"]])
display(productos.assign(ranking_precio=productos["precio_unitario"].rank(ascending=False, method="min").astype(int))
        .sort_values("ranking_precio")[["producto", "precio_unitario", "ranking_precio"]].head(5))

## 4.4 Transformación
Crear columnas nuevas o **convertir** las existentes:
`astype` (tipo), `map` (diccionario), `np.where` (condición), `pd.cut` (rangos), `.str` (texto), `.dt` (fechas).

In [ ]:
p = pedidos.copy()
p["mes"] = p["fecha_pedido"].dt.to_period("M").astype(str)                    # componentes de fecha
p["dia_semana"] = p["fecha_pedido"].dt.day_name()
p["tamano_pedido"] = pd.cut(p["cantidad"], bins=[0, 2, 4, 100], labels=["Pequeño", "Mediano", "Grande"])
p["con_descuento"] = np.where(p["descuento"].fillna(0) > 0, "Sí", "No")        # condición
p["canal_pago"] = p["metodo_pago"].map({"Tarjeta": "Digital", "Transferencia": "Digital", "Efectivo": "Físico"})
p["estado"] = p["estado"].astype("category")                                    # cambio de tipo

display(p[["fecha_pedido", "mes", "dia_semana", "cantidad", "tamano_pedido", "con_descuento", "canal_pago"]].head())

# ¿Por qué usar 'category'? Ahorra memoria en columnas con pocos valores repetidos
ciudad_obj = clientes["ciudad"].memory_usage(deep=True)
ciudad_cat = clientes["ciudad"].astype("category").memory_usage(deep=True)
print(f"Memoria 'ciudad' como texto: {ciudad_obj/1024:.1f} KB  vs  como category: {ciudad_cat/1024:.1f} KB")

## 4.5 Limpieza
Los datos reales traen **errores**: duplicados, texto inconsistente, nulos, valores atípicos y tipos incorrectos.
Cada decisión (borrar, imputar, corregir) debe ser **consciente y justificada**.

In [ ]:
cl = clientes.copy()

# 1) Duplicados
print("Duplicados exactos:", int(cl.duplicated().sum()))
cl = cl.drop_duplicates()

# 2) Texto inconsistente
print("Variantes de ciudad ANTES:", cl["ciudad"].nunique())
cl["ciudad"] = cl["ciudad"].str.strip().str.title()
print("Variantes de ciudad DESPUÉS:", cl["ciudad"].nunique(), "→", sorted(cl["ciudad"].unique()))

# 3) Tipo incorrecto: fecha guardada como texto
cl["fecha_registro"] = pd.to_datetime(cl["fecha_registro"], format="%d/%m/%Y")

# 4) Nulos: imputar (edad) o descartar (email)
print("Nulos edad:", int(cl["edad"].isna().sum()), "| mediana:", cl["edad"].median())
cl["edad"] = cl["edad"].fillna(cl["edad"].median())
print("Clientes con email:", int(cl["email"].notna().sum()), "de", len(cl))

# 5) Valores atípicos con la regla del IQR
q1, q3 = pedidos["cantidad"].quantile([.25, .75]); iqr = q3 - q1
atipicos = pedidos[(pedidos["cantidad"] < q1 - 1.5 * iqr) | (pedidos["cantidad"] > q3 + 1.5 * iqr)]
print(f"\nLímites IQR: [{q1 - 1.5*iqr:.1f}, {q3 + 1.5*iqr:.1f}] → atípicos encontrados:")
display(atipicos[["id_pedido", "cantidad"]])

## 4.6 Combinación de tablas
- **`merge`** une por una clave (como `JOIN`): `inner`, `left`, `right`, `outer`.
- **`concat`** apila tablas con las mismas columnas (como `UNION`).
- `validate="m:1"` verifica la relación esperada y evita **filas duplicadas por error**.

In [ ]:
ped = pedidos.merge(productos[["id_producto", "producto", "categoria", "precio_unitario"]],
                    on="id_producto", how="left", validate="m:1")
ped["ingreso"] = ped["cantidad"] * ped["precio_unitario"] * (1 - ped["descuento"].fillna(0))
display(ped[["id_pedido", "producto", "categoria", "cantidad", "precio_unitario", "descuento", "ingreso"]].head())

print("Filas antes y después del merge:", len(pedidos), "→", len(ped), "(deben coincidir)")

# Tipos de join: ¿cuántos productos NO se han pedido nunca?
resumen = productos.merge(pedidos.groupby("id_producto").size().rename("n_pedidos"),
                          on="id_producto", how="left")
print("Productos sin pedidos:", int(resumen["n_pedidos"].isna().sum()))

## 4.7 Agregación
Resumir muchos registros en pocos números: **dividir en grupos → aplicar una función → combinar** (`GROUP BY`).

In [ ]:
display(pedidos["metodo_pago"].value_counts(normalize=True).round(3).to_frame("proporción"))

por_categoria = (ped[ped["estado"] != "Cancelado"]
                 .groupby("categoria")
                 .agg(pedidos=("id_pedido", "count"),
                      unidades=("cantidad", "sum"),
                      ingreso_total=("ingreso", "sum"),
                      ticket_medio=("ingreso", "mean"))
                 .round(2).sort_values("ingreso_total", ascending=False))
display(por_categoria)

por_categoria["ingreso_total"].plot.bar(color="#4C78A8", rot=0, title="Ingreso total por categoría (sin cancelados)")
plt.show()

## 4.8 Reestructuración
Cambiar la **forma** de la tabla: de larga a ancha (`pivot_table`, `crosstab`) y de ancha a larga (`melt`).

In [ ]:
ok = ped[ped["estado"] != "Cancelado"]
ancha = ok.pivot_table(index="categoria", columns="metodo_pago", values="ingreso", aggfunc="sum").round(0)
display(ancha)

larga = ancha.reset_index().melt(id_vars="categoria", var_name="metodo_pago", value_name="ingreso")
display(larga.head(4))

display(pd.crosstab(ped["estado"], ped["metodo_pago"], normalize="index").round(3))

## 4.9 Series de tiempo y ventanas
Operaciones que dependen del **orden**: re-muestreo, acumulados, variaciones y medias móviles.

In [ ]:
mensual = ok.set_index("fecha_pedido")["ingreso"].resample("MS").sum().to_frame("ingreso")
mensual["acumulado"] = mensual["ingreso"].cumsum()
mensual["variacion_%"] = (mensual["ingreso"].pct_change() * 100).round(1)
mensual["media_movil_3m"] = mensual["ingreso"].rolling(3).mean().round(0)
mensual.index = mensual.index.strftime("%Y-%m")
display(mensual.round(0).head(6))

mensual[["ingreso", "media_movil_3m"]].plot(marker="o", title="Ingreso mensual y media móvil de 3 meses")
plt.xticks(rotation=45); plt.show()

## 4.10 Exportación
Guardar el resultado para compartirlo o usarlo en otra herramienta.

In [ ]:
por_categoria.to_csv("resumen_por_categoria.csv", encoding="utf-8")
with pd.ExcelWriter("resumen_ventas.xlsx") as w:
    por_categoria.to_excel(w, sheet_name="por_categoria")
    mensual.to_excel(w, sheet_name="mensual")
print("✅ Archivos creados:", [f for f in os.listdir(".") if f.startswith("resumen_")])

# 5 · Equivalencias entre herramientas

| Operación | pandas | SQL | Excel |
|---|---|---|---|
| Elegir columnas | `df[["a","b"]]` | `SELECT a, b` | ocultar/copiar columnas |
| Filtrar filas | `df[df.a > 5]` | `WHERE a > 5` | Filtro automático |
| Ordenar | `sort_values("a")` | `ORDER BY a` | Ordenar |
| Columna calculada | `df["c"] = df.a * df.b` | `SELECT a*b AS c` | fórmula `=A2*B2` |
| Quitar duplicados | `drop_duplicates()` | `SELECT DISTINCT` | Quitar duplicados |
| Rellenar nulos | `fillna(0)` | `COALESCE(a, 0)` | `=SI.ND()` / `=SI(ESBLANCO())` |
| Unir tablas | `merge(..., on="id")` | `JOIN ... ON` | `BUSCARV` / `XLOOKUP` |
| Agrupar | `groupby("a").sum()` | `GROUP BY a` | Tabla dinámica |
| Tabla dinámica | `pivot_table(...)` | `CASE` + `GROUP BY` | Tabla dinámica |
| Acumulado | `cumsum()` | `SUM() OVER (...)` | suma acumulada con fórmula |

# 6 · Ejercicios

**Reglas**
- Escribe tu código donde dice `# TU CÓDIGO AQUÍ` y deja el resultado en la variable `respuesta`.
- Ejecuta la celda siguiente (`verificar`) para **autocorregirte**: te dirá ✅ o ❌ sin mostrarte la solución.
- Los ejercicios 4 a 7 reutilizan variables de ejercicios anteriores (`ped_ok`, `c`), así que resuélvelos **en orden**.
- No modifiques las tablas originales: trabaja siempre sobre copias (`.copy()`).

In [ ]:
ESPERADOS = {
 "ej1": {
  "nulos_edad": 32,
  "nulos_email": 51,
  "fecha_min": "2024-01-01"
 },
 "ej2": 428,
 "ej3": {
  "filas": 500,
  "ciudades": 8
 },
 "ej4": 49857.82,
 "ej5": {
  "categoria": "Platos fuertes",
  "ingreso": 29635.31
 },
 "ej6": {
  "ciudad": "Cuenca",
  "pedidos": 195
 },
 "ej7": {
  "grupo": "36-45",
  "ticket": 19.57
 }
}


def _igual(a, b):
    if isinstance(b, (int, float)) and not isinstance(b, bool):
        try:
            return abs(float(a) - float(b)) <= 0.011
        except Exception:
            return False
    return str(a).strip() == str(b).strip()


def verificar(clave, resp):
    esp = ESPERADOS[clave]
    vacio = resp is None or (isinstance(resp, dict) and any(v is None for v in resp.values()))
    if vacio:
        print("✏️ Aún no hay respuesta: completa el código y la variable 'respuesta'.")
        return
    if isinstance(esp, dict):
        malos = [k for k in esp if not _igual(resp.get(k), esp[k])]
    else:
        malos = [] if _igual(resp, esp) else ["valor"]
    if malos:
        print("❌ Todavía no coincide. Revisa:", ", ".join(malos), "· (usa la pista del enunciado)")
    else:
        print("✅ ¡Correcto!")

## Ejercicio 1 · Inspección y tipos de datos  ⭐

Trabaja con la tabla **`clientes`** tal como viene del Excel (sin limpiar).

1. Cuenta los valores nulos de `edad` y de `email`.
2. Convierte `fecha_registro` (texto `dd/mm/aaaa`) a tipo fecha.
3. Obtén la **fecha mínima** de registro (como texto `AAAA-MM-DD`).

<details><summary>💡 Pista</summary>

`isna().sum()`, `pd.to_datetime(..., format="%d/%m/%Y")`, `.min().date()` y `str(...)`.

</details>

In [ ]:
c = clientes.copy()
# TU CÓDIGO AQUÍ

respuesta = {"nulos_edad": None, "nulos_email": None, "fecha_min": None}

verificar("ej1", respuesta)

## Ejercicio 2 · Filtrado con condiciones combinadas  ⭐

En **`pedidos`**, ¿cuántas filas cumplen **a la vez**:
- `estado` igual a `"Entregado"`,
- `cantidad` mayor o igual a 3,
- `descuento` mayor que 0.10?

Respuesta: un número entero.

<details><summary>💡 Pista</summary>

Combina condiciones con `&` y **paréntesis** en cada una. Los nulos de `descuento` no cumplen `> 0.1`.

</details>

In [ ]:
# TU CÓDIGO AQUÍ

respuesta = None

verificar("ej2", respuesta)

## Ejercicio 3 · Limpieza de datos  ⭐⭐

En **`clientes`**:

1. Elimina los clientes duplicados (mismo `id_cliente`).
2. Normaliza `ciudad`: sin espacios sobrantes y con formato Título (`"quito "` → `"Quito"`).

Guarda el resultado en la variable **`c`** (la usarás en los ejercicios 6 y 7).
Respuesta: número de filas y número de ciudades distintas.

<details><summary>💡 Pista</summary>

`drop_duplicates(subset=...)`, `.str.strip().str.title()` y `nunique()`.

</details>

In [ ]:
c = clientes.copy()
# TU CÓDIGO AQUÍ

respuesta = {"filas": None, "ciudades": None}

verificar("ej3", respuesta)

## Ejercicio 4 · Combinar tablas y crear columnas  ⭐⭐

1. Une **`pedidos`** con **`productos`** (por `id_producto`) trayendo `categoria` y `precio_unitario`.
2. Crea la columna **`ingreso`** = `cantidad × precio_unitario × (1 − descuento)`. Un descuento nulo significa **sin descuento** (0).
3. Quédate solo con los pedidos que **no** estén `"Cancelado"`. Guarda ese DataFrame en **`ped_ok`**.

Respuesta: ingreso total de `ped_ok`, redondeado a 2 decimales.

<details><summary>💡 Pista</summary>

`merge(..., how="left", validate="m:1")`, `fillna(0)` y filtra con `!=`.

</details>

In [ ]:
# TU CÓDIGO AQUÍ
ped_ok = None

respuesta = None

verificar("ej4", respuesta)

## Ejercicio 5 · Agregación (group by)  ⭐⭐

Con **`ped_ok`** (ejercicio 4): ¿qué **categoría** genera más ingreso y cuánto?

Respuesta: nombre de la categoría e ingreso (2 decimales).

<details><summary>💡 Pista</summary>

`groupby("categoria")["ingreso"].sum()` y luego `idxmax()` / `max()`.

</details>

In [ ]:
# Usa ped_ok del ejercicio 4
# TU CÓDIGO AQUÍ

respuesta = {"categoria": None, "ingreso": None}

verificar("ej5", respuesta)

## Ejercicio 6 · Tabla dinámica (pivot)  ⭐⭐⭐

Une **`ped_ok`** con la tabla limpia **`c`** (ejercicio 3) para conocer la `ciudad` de cada pedido y arma una tabla dinámica: ciudades en filas, `metodo_pago` en columnas, **número de pedidos** en los valores.

Respuesta: la ciudad con más pedidos pagados con `"Tarjeta"` y cuántos son.

<details><summary>💡 Pista</summary>

`merge` con `c[["id_cliente", "ciudad"]]`, luego `pivot_table(index=..., columns=..., values="id_pedido", aggfunc="count")`.

</details>

In [ ]:
# Usa ped_ok (ej. 4) y c (ej. 3)
# TU CÓDIGO AQUÍ

respuesta = {"ciudad": None, "pedidos": None}

verificar("ej6", respuesta)

## Ejercicio 7 · Reto: segmentación por grupos de edad  ⭐⭐⭐⭐

**Reto integrador.** Con `c` (ej. 3) y `ped_ok` (ej. 4):

1. Crea `grupo_edad` en `c` con `pd.cut` usando los cortes `[17, 25, 35, 45, 55, 70]` y las etiquetas `"18-25"`, `"26-35"`, `"36-45"`, `"46-55"`, `"56-70"`.
2. Une con `ped_ok` y calcula el **ticket medio** (promedio de `ingreso` por pedido) de cada grupo. Los clientes sin edad se descartan.

Respuesta: el grupo con mayor ticket medio y su valor (2 decimales).

<details><summary>💡 Pista</summary>

`pd.cut` deja los intervalos abiertos a la izquierda: `(17, 25]` incluye el 18 y el 25. Usa `groupby(..., observed=True)`.

</details>

In [ ]:
# Usa c (ej. 3) y ped_ok (ej. 4)
# TU CÓDIGO AQUÍ

respuesta = {"grupo": None, "ticket": None}

verificar("ej7", respuesta)

# ✅ Cierre

**Lo esencial**
- Un dato estructurado tiene **esquema fijo**: tablas, filas, columnas tipadas y claves.
- El **tipo de cada columna** decide qué operaciones son válidas.
- Flujo de trabajo habitual: **inspeccionar → limpiar → transformar → combinar → agregar → comunicar**.
- Casi toda operación en pandas tiene un equivalente directo en **SQL** y en **Excel**.

**Para seguir practicando:** repite los ejercicios usando el dataset de Zomato del notebook anterior.